# 05 · Experimento V4 · Contexto espacial · CV temporal interna

Proyecto: **Clasificación anticipada de amenaza meteorológica por lluvias intensas**.

Objetivo del notebook: comparar, bajo las mismas particiones temporales y características, los cinco modelos definidos en la Tarea #4:

1. Random Forest
2. Regresión Logística Multinomial
3. XGBoost
4. SVM con kernel RBF
5. MLP

Además se incluye un `DummyClassifier` únicamente como baseline de referencia.

**Regla metodológica:** el conjunto de prueba temporal y el holdout espacial no se utilizan para ajustar hiperparámetros ni para elegir el modelo ganador.

## 0. Dependencias

In [1]:
# Ejecutar una sola vez si alguna dependencia no está instalada.
# En Colab suele bastar con ejecutar esta celda.
%pip install -q scikit-learn xgboost joblib matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importaciones y configuración

In [2]:
from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
    recall_score,
)
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Entorno listo.")

Entorno listo.


## 2. Ruta del proyecto

El notebook intenta detectar automáticamente la carpeta del proyecto. Si no la encuentra, cambia `PROJECT_DIR` manualmente.

In [3]:
candidates = [
    Path.cwd(),
    Path.cwd() / "rain-threat-classifier",
    Path("/content/rain-threat-classifier"),
    Path("/content/drive/MyDrive/rain-threat-classifier"),
]

PROJECT_DIR = next(
    (p for p in candidates if (p / "resultados_completo" / "dataset_modelo_mensual_v4.csv").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró el proyecto. Define PROJECT_DIR con la ruta de rain-threat-classifier."
    )

DATA_DIR = PROJECT_DIR / "resultados_completo"
EXPERIMENT_NAME = "v4_espacial_182_cv"
OUTPUT_DIR = PROJECT_DIR / "resultados_experimentos" / EXPERIMENT_NAME
ARTIFACT_DIR = PROJECT_DIR / "artefactos_experimentos" / EXPERIMENT_NAME

for folder in [OUTPUT_DIR, ARTIFACT_DIR, OUTPUT_DIR / "matrices_confusion", OUTPUT_DIR / "predicciones"]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)

PROJECT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier


## 3. Carga del dataset V4 y sus características

In [4]:
DATASET_PATH = DATA_DIR / "dataset_modelo_mensual_v4.csv"
FEATURES_PATH = DATA_DIR / "columnas_modelo_v4.txt"

df = pd.read_csv(DATASET_PATH)
df["period_start"] = pd.to_datetime(df["period_start"])
df["target_period_start"] = pd.to_datetime(df["target_period_start"])

features = [
    line.strip()
    for line in FEATURES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Dataset:", df.shape)
print("Características V4:", len(features))
print("Columnas faltantes:", sorted(set(features) - set(df.columns)))

Dataset: (6120, 224)
Características V4: 182
Columnas faltantes: []


In [5]:
# Auditoría específica de V4
V4_TEMPORAL_FEATURES = [
    "prcptot_mm_lag11",
    "rx1day_mm_lag11",
    "rx5day_mm_lag11",
    "max_6h_mm_lag11",
    "wet_days_lag11",
    "sdii_mm_per_wet_day_lag11",
    "temperature_mean_c_lag11",
    "relative_humidity_mean_pct_lag11",
    "prcptot_media_3m",
    "prcptot_media_6m",
    "rx5_media_3m",
    "rx5_max_6m",
    "humedad_media_3m",
    "temperatura_media_3m",
    "prcptot_tendencia_1m",
    "rx5_tendencia_1m",
    "humedad_tendencia_1m",
    "temperatura_tendencia_1m",
]

assert set(V4_TEMPORAL_FEATURES).issubset(df.columns)
assert "target_amenaza" not in features
assert "target_rx5day_mm" not in features
assert df[features].isna().sum().sum() == 0

print("V4 validada. Características temporales V4 presentes:", len(V4_TEMPORAL_FEATURES))


V4 validada. Características temporales V4 presentes: 18


In [6]:
# Auditoría específica de contexto espacial V4
zone_features = [c for c in features if c.startswith("zona_")]
region_features = [c for c in features if c.startswith("region_")]

print("Features zona:", len(zone_features), zone_features)
print("Features región:", len(region_features), region_features)

assert len(zone_features) == 12, f"Se esperaban 12 features zona y llegaron {len(zone_features)}"
assert len(region_features) == 3, f"Se esperaban 3 features región y llegaron {len(region_features)}"

train_rows_check = df[df["split"] == "entrenamiento"]
assert (train_rows_check[zone_features].sum(axis=1) == 1).all()
assert (train_rows_check[region_features].sum(axis=1) == 1).all()

print("Contexto espacial V4 validado.")


Features zona: 12 ['zona_babahoyo', 'zona_cuenca', 'zona_esmeraldas', 'zona_guayaquil', 'zona_loja', 'zona_machala', 'zona_portoviejo', 'zona_puyo', 'zona_quito', 'zona_riobamba', 'zona_salinas', 'zona_tena']
Features región: 3 ['region_amazonia', 'region_litoral', 'region_sierra']
Contexto espacial V4 validado.


## 4. Auditoría rápida antes de entrenar

In [7]:
assert df.shape[0] == 6120, f"Se esperaban 6120 filas en V4 y llegaron {len(df)}"
assert len(features) == 182, f"Se esperaban 182 características V4 y llegaron {len(features)}"
assert set(features).issubset(df.columns)
assert df[features].isna().sum().sum() == 0, "Hay nulos en las características V4."
assert df["target_amenaza"].isna().sum() == 0, "Hay targets nulos."
assert set(df["target_amenaza"].unique()) == {"Baja", "Media", "Alta"}

print("Distribución por partición (solo para auditoría de conteos):")
display(df["split"].value_counts().rename("filas").to_frame())

print("IMPORTANTE: para decidir si V4 sirve, a partir de aquí se usa SOLO split=entrenamiento.")


Distribución por partición (solo para auditoría de conteos):


,filas
split,
entrenamiento,3744
holdout_historia,936
validacion_temporal,576
prueba_temporal,576
holdout_espacial,288


IMPORTANTE: para decidir si V4 sirve, a partir de aquí se usa SOLO split=entrenamiento.


## 5. Aislar exclusivamente el conjunto de entrenamiento

- `entrenamiento`: 12 zonas, objetivos hasta 2017. Se usa para aprender y ajustar hiperparámetros.
- `validacion_temporal`: 2018–2021. Se usa para comparar los cinco modelos ya ajustados.
- `prueba_temporal`: 2022–2025. Se reserva para la evaluación final.
- `holdout_espacial`: Santo Domingo, Nueva Loja y Macas en 2018–2025. Se reserva para evaluar generalización geográfica.
- `holdout_historia`: historia de las zonas reservadas. **No se usa para ajustar el clasificador.**

In [8]:
train_df = df[df["split"] == "entrenamiento"].copy()
X_train = train_df[features].copy()

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["target_amenaza"])
CLASS_NAMES = list(label_encoder.classes_)

print("Orden interno de clases:", dict(enumerate(CLASS_NAMES)))
print("Entrenamiento V4:", X_train.shape)
print("Target desde:", train_df["target_period_start"].min())
print("Target hasta:", train_df["target_period_start"].max())


Orden interno de clases: {0: 'Alta', 1: 'Baja', 2: 'Media'}
Entrenamiento V4: (3744, 182)
Target desde: 1992-01-01 00:00:00
Target hasta: 2017-12-01 00:00:00


## 6. Validación cruzada temporal interna

No se usa K-Fold aleatorio. Los folds respetan el orden cronológico y todas las zonas de un mismo periodo objetivo permanecen juntas.

Estos folds solo utilizan `entrenamiento` (hasta 2017). La validación 2018–2021 sigue intacta.

In [9]:
TEMPORAL_FOLDS = [
    ("F1", "2004-12-01", "2005-01-01", "2007-12-01"),
    ("F2", "2007-12-01", "2008-01-01", "2010-12-01"),
    ("F3", "2010-12-01", "2011-01-01", "2013-12-01"),
    ("F4", "2013-12-01", "2014-01-01", "2017-12-01"),
]

def build_temporal_folds(frame):
    folds = []
    description = []
    dates = frame["target_period_start"]

    for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
        train_mask = dates <= pd.Timestamp(train_end)
        val_mask = dates.between(pd.Timestamp(val_start), pd.Timestamp(val_end))

        train_idx = np.flatnonzero(train_mask.to_numpy())
        val_idx = np.flatnonzero(val_mask.to_numpy())
        folds.append((train_idx, val_idx))
        description.append({
            "fold": name,
            "train_filas": len(train_idx),
            "val_filas": len(val_idx),
            "train_hasta": train_end,
            "val_desde": val_start,
            "val_hasta": val_end,
        })

    return folds, pd.DataFrame(description)

temporal_folds, folds_table = build_temporal_folds(train_df)
display(folds_table)

,fold,train_filas,val_filas,train_hasta,val_desde,val_hasta
0,F1,1872,432,2004-12-01,2005-01-01,2007-12-01
1,F2,2304,432,2007-12-01,2008-01-01,2010-12-01
2,F3,2736,432,2010-12-01,2011-01-01,2013-12-01
3,F4,3168,576,2013-12-01,2014-01-01,2017-12-01


## 7. Funciones de evaluación

In [10]:
def calculate_metrics(y_true, y_pred, class_names=CLASS_NAMES):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(class_names)), zero_division=0
    )

    metrics = {
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_acc,
    }
    for i, class_name in enumerate(class_names):
        key = class_name.lower()
        metrics[f"precision_{key}"] = precision[i]
        metrics[f"recall_{key}"] = recall[i]
        metrics[f"f1_{key}"] = f1[i]
        metrics[f"support_{key}"] = int(support[i])
    return metrics

def show_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    return fig

def evaluate_model(model, X, y):
    start = time.perf_counter()
    pred = model.predict(X)
    inference_seconds = time.perf_counter() - start
    metrics = calculate_metrics(y, pred)
    metrics["inference_seconds"] = inference_seconds
    return metrics, pred

## 8. Baselines internos sobre los mismos folds temporales

Para no consultar todavía 2018–2021, Dummy y Persistencia se evalúan exclusivamente dentro de los cuatro folds de 1991–2017.


In [11]:
baseline_rows = []

for fold_id, (train_idx, val_idx) in enumerate(temporal_folds, start=1):
    # Dummy: clase más frecuente del sub-train del fold
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train.iloc[train_idx], y_train[train_idx])
    dummy_pred = dummy.predict(X_train.iloc[val_idx])
    dummy_m = calculate_metrics(y_train[val_idx], dummy_pred)
    baseline_rows.append({
        "baseline": "Dummy", "fold": f"F{fold_id}", **dummy_m
    })

    # Persistencia: amenaza(t) como estimación de amenaza(t+1)
    persistence_pred = label_encoder.transform(
        train_df.iloc[val_idx]["amenaza_mes"]
    )
    persistence_m = calculate_metrics(y_train[val_idx], persistence_pred)
    baseline_rows.append({
        "baseline": "Persistencia", "fold": f"F{fold_id}", **persistence_m
    })

baseline_fold_table = pd.DataFrame(baseline_rows)
baseline_cv = (
    baseline_fold_table.groupby("baseline")
    .agg(
        macro_f1_cv_mean=("macro_f1", "mean"),
        macro_f1_cv_std=("macro_f1", "std"),
        balanced_accuracy_cv_mean=("balanced_accuracy", "mean"),
        recall_alta_cv_mean=("recall_alta", "mean"),
    )
    .sort_values("macro_f1_cv_mean", ascending=False)
)
display(baseline_cv)


,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean
baseline,,,,
Persistencia,0.369561,0.031969,0.369602,0.349616
Dummy,0.155065,0.016693,0.333333,0.250000


In [ ]:
# Persistencia ya fue evaluada dentro de los cuatro folds temporales.
print("Baselines temporales internos listos.")


## 9. Modelos y espacios pequeños de hiperparámetros

Los modelos sensibles a escala (`Logística`, `SVM`, `MLP`) incluyen `StandardScaler` dentro de un `Pipeline`. Random Forest y XGBoost no requieren escalado.

In [12]:
models = {
    "Regresion_Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)),
    ]),
    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
    ]),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            max_iter=600,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_STATE,
        )),
    ]),
}

param_spaces = {
    "Regresion_Logistica": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
        "model__class_weight": [None, "balanced"],
    },
    "Random_Forest": {
        "n_estimators": [250, 400, 600],
        "max_depth": [None, 10, 18, 26],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5],
        "class_weight": [None, "balanced"],
    },
    "XGBoost": {
        "n_estimators": [200, 350, 500],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.06, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
    },
    "SVM_RBF": {
        "model__C": [0.1, 1.0, 10.0, 30.0],
        "model__gamma": ["scale", 0.01, 0.001],
        "model__class_weight": [None, "balanced"],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64,), (128, 64), (128, 64, 32)],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.0005, 0.001, 0.003],
    },
}

print("Modelos definidos:", list(models))

Modelos definidos: ['Regresion_Logistica', 'Random_Forest', 'XGBoost', 'SVM_RBF', 'MLP']


## 10. Búsqueda temporal de hiperparámetros

Se usa una búsqueda aleatoria pequeña para mantener el tiempo de ejecución razonable. Cada configuración se evalúa en los cuatro folds temporales y se selecciona por **Macro F1 promedio**.

In [13]:
def temporal_random_search(
    model_name,
    estimator,
    param_space,
    X,
    y,
    folds,
    n_iter=8,
    random_state=RANDOM_STATE,
):
    sampled_params = list(ParameterSampler(param_space, n_iter=n_iter, random_state=random_state))
    rows = []

    for config_id, params in enumerate(sampled_params, start=1):
        fold_scores = []
        started = time.perf_counter()

        for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
            candidate = clone(estimator).set_params(**params)
            candidate.fit(X.iloc[train_idx], y[train_idx])
            pred = candidate.predict(X.iloc[val_idx])
            fold_scores.append(f1_score(y[val_idx], pred, average="macro"))

        rows.append({
            "modelo": model_name,
            "config_id": config_id,
            "macro_f1_cv_mean": float(np.mean(fold_scores)),
            "macro_f1_cv_std": float(np.std(fold_scores)),
            "seconds": time.perf_counter() - started,
            "params": params,
        })
        print(
            f"{model_name} | {config_id:02d}/{len(sampled_params)} | "
            f"Macro F1={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}"
        )

    result = pd.DataFrame(rows).sort_values(
        ["macro_f1_cv_mean", "macro_f1_cv_std"],
        ascending=[False, True],
    ).reset_index(drop=True)
    return result

# Para una primera ejecución rápida, 6 configuraciones por modelo es suficiente.
# Si sobra tiempo, subir a 10-15.
N_ITER = 6

search_results = {}
for model_name, estimator in models.items():
    print("\n" + "=" * 80)
    print("AJUSTANDO:", model_name)
    search_results[model_name] = temporal_random_search(
        model_name,
        estimator,
        param_spaces[model_name],
        X_train,
        y_train,
        temporal_folds,
        n_iter=N_ITER,
    )

all_search_results = pd.concat(search_results.values(), ignore_index=True)
all_search_results.to_csv(OUTPUT_DIR / "busqueda_hiperparametros.csv", index=False)
print("Guardado:", OUTPUT_DIR / "busqueda_hiperparametros.csv")


AJUSTANDO: Regresion_Logistica
Regresion_Logistica | 01/6 | Macro F1=0.3297 ± 0.0102
Regresion_Logistica | 02/6 | Macro F1=0.3340 ± 0.0075
Regresion_Logistica | 03/6 | Macro F1=0.3316 ± 0.0117
Regresion_Logistica | 04/6 | Macro F1=0.3322 ± 0.0141
Regresion_Logistica | 05/6 | Macro F1=0.3225 ± 0.0103
Regresion_Logistica | 06/6 | Macro F1=0.3246 ± 0.0155

AJUSTANDO: Random_Forest
Random_Forest | 01/6 | Macro F1=0.3178 ± 0.0215
Random_Forest | 02/6 | Macro F1=0.3268 ± 0.0242
Random_Forest | 03/6 | Macro F1=0.3255 ± 0.0096
Random_Forest | 04/6 | Macro F1=0.3203 ± 0.0197
Random_Forest | 05/6 | Macro F1=0.3238 ± 0.0182
Random_Forest | 06/6 | Macro F1=0.3206 ± 0.0169

AJUSTANDO: XGBoost
XGBoost | 01/6 | Macro F1=0.3191 ± 0.0158
XGBoost | 02/6 | Macro F1=0.3264 ± 0.0286
XGBoost | 03/6 | Macro F1=0.3246 ± 0.0343
XGBoost | 04/6 | Macro F1=0.3279 ± 0.0238
XGBoost | 05/6 | Macro F1=0.3391 ± 0.0202
XGBoost | 06/6 | Macro F1=0.3246 ± 0.0267

AJUSTANDO: SVM_RBF
SVM_RBF | 01/6 | Macro F1=0.3136 ± 0.0

## 11. Mejor configuración por modelo

In [14]:
best_params = {}
for name, result in search_results.items():
    row = result.iloc[0]
    best_params[name] = row["params"]
    print(f"{name}: Macro F1 CV={row['macro_f1_cv_mean']:.4f} | {row['params']}")

with open(OUTPUT_DIR / "mejores_hiperparametros.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False, default=str)

Regresion_Logistica: Macro F1 CV=0.3340 | {'model__class_weight': 'balanced', 'model__C': 0.01}
Random_Forest: Macro F1 CV=0.3268 | {'n_estimators': 250, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 26, 'class_weight': 'balanced'}
XGBoost: Macro F1 CV=0.3391 | {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
SVM_RBF: Macro F1 CV=0.3498 | {'model__gamma': 0.01, 'model__class_weight': 'balanced', 'model__C': 10.0}
MLP: Macro F1 CV=0.3463 | {'model__learning_rate_init': 0.003, 'model__hidden_layer_sizes': (64,), 'model__alpha': 0.001}


## 12. Comparación interna de V4

Esta tabla es la única que debe utilizarse por ahora para decidir si V4 merece pasar a la validación externa 2018–2021. **No se ha consultado esa partición.**


In [15]:
cv_rows = []
for name, result in search_results.items():
    best = result.iloc[0]
    cv_rows.append({
        "modelo": name,
        "macro_f1_cv_mean": best["macro_f1_cv_mean"],
        "macro_f1_cv_std": best["macro_f1_cv_std"],
    })

cv_model_table = pd.DataFrame(cv_rows)

baseline_display = baseline_cv.reset_index()[
    ["baseline", "macro_f1_cv_mean", "macro_f1_cv_std"]
].rename(columns={"baseline": "modelo"})

cv_comparison = pd.concat(
    [cv_model_table, baseline_display], ignore_index=True
).sort_values("macro_f1_cv_mean", ascending=False).reset_index(drop=True)

display(cv_comparison)
cv_comparison.to_csv(OUTPUT_DIR / "comparacion_cv_interna_v4.csv", index=False)

print("\nNO ejecutar todavía validación temporal 2018–2021.")


,modelo,macro_f1_cv_mean,macro_f1_cv_std
0,Persistencia,0.369561,0.031969
1,SVM_RBF,0.349758,0.014791
2,MLP,0.346253,0.007253
3,XGBoost,0.339061,0.020162
4,Regresion_Logistica,0.333966,0.007455
5,Random_Forest,0.326809,0.024188
6,Dummy,0.155065,0.016693



NO ejecutar todavía validación temporal 2018–2021.
